<a href="https://colab.research.google.com/github/abdur004/PDF-Summarizer-T5/blob/main/T5tr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🔧 Clean start: uninstall wandb to avoid unwanted logging
!pip uninstall -y wandb


Found existing installation: wandb 0.19.9
Uninstalling wandb-0.19.9:
  Successfully uninstalled wandb-0.19.9


In [ ]:
!pip install transformers datasets torch PyMuPDF sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import fitz  # PyMuPDF for PDFs
from datasets import load_dataset, Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments


In [ ]:
# Load the CNN/DailyMail dataset (only 1% to prevent memory issues)
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train").shuffle(seed=42).select(range(int(0.01 * len(load_dataset("cnn_dailymail", "3.0.0", split="train")))))

# Convert dataset to required format
documents = [{"text": data["article"], "summary": data["highlights"]} for data in dataset]

# Convert to Hugging Face Dataset format
dataset = Dataset.from_list(documents)

# Split into train (80%) and test (20%) sets
dataset = dataset.train_test_split(test_size=0.2)
train_dataset, test_dataset = dataset["train"], dataset["test"]


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
# Load tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

def preprocess_function(sample):
    """Tokenizes text and summaries."""
    model_inputs = tokenizer(sample["text"], max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(sample["summary"], max_length=150, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply tokenization
train_dataset = train_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/2296 [00:00<?, ? examples/s]

Map:   0%|          | 0/575 [00:00<?, ? examples/s]

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

training_args = TrainingArguments(
    output_dir="./t5_summarizer",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=2,  # Reduce batch size to fit within 15GB
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  # Accumulate gradients to simulate larger batch size
    learning_rate=3e-4,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=2,  # Reduce epochs to prevent memory overflow
    fp16=True,  # Enable mixed precision to optimize memory
    logging_dir="./logs",
)


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,No log,0.937704
2,1.139100,0.937187


TrainOutput(global_step=574, training_loss=1.1151792812015122, metrics={'train_runtime': 228.481, 'train_samples_per_second': 20.098, 'train_steps_per_second': 2.512, 'total_flos': 621489552359424.0, 'train_loss': 1.1151792812015122, 'epoch': 2.0})

In [ ]:
model.save_pretrained("./fine_tuned_t5_summarizer")
tokenizer.save_pretrained("./fine_tuned_t5_summarizer")


('./fine_tuned_t5_summarizer/tokenizer_config.json',
 './fine_tuned_t5_summarizer/special_tokens_map.json',
 './fine_tuned_t5_summarizer/spiece.model',
 './fine_tuned_t5_summarizer/added_tokens.json')

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# Path to your fine-tuned model
model_path = "./t5_summarizer/checkpoint-574"

# Load the original tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")  # or "t5-base", whichever you used

# Load the fine-tuned model
model = T5ForConditionalGeneration.from_pretrained(model_path)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
def summarize_text(text, max_input_length=512, max_output_length=150):
    # Tokenize input
    inputs = tokenizer.encode(
        "summarize: " + text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    ).to(device)

    # Generate summary ids
    summary_ids = model.generate(
        inputs,
        max_length=max_output_length,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )

    # Decode and return the summary
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [ ]:
sample_text = (
    "In a small, dimly lit studio at the heart of the city, an old architect named Elias sat at his desk, tracing"
     "the edges of a yellowed sketch with his fingertips. The paper was fragile, worn by time, but the design it held was timeless—an elegant structure that had never been built.Decades ago, Elias had poured his heart into this design. It was meant to be a revolutionary library, asanctuary for those who sought knowledge. The city, however, had chosen a more conventional"
     "design, and Elias had buried the sketch away, along with his youthful dreams.But tonight, something stirred within him. The world had changed, and so had architecture. Peoplenow sought innovation, creativity, and emotion in their spaces. What if this forgotten design couldfinally find its place?Determined, Elias spent the next few days refining his old work, adapting it to modern needs while"
     "preserving its soul. He reached out to an old friend who now worked with a firm renowned for embracing daring ideas. The friend was captivated and arranged a presentation.A week later, Elias stood before a panel of investors, his hands steady as he unveiled the design. Ashe spoke, he saw curiosity spark in their eyes, followed by admiration. When he finished, the room"
     "erupted in applause.Months passed, and the foundation for the library was laid. As Elias watched the first stones being placed, he felt something he hadn’t in years—a sense of fulfillment. His forgotten sketch was forgotten no more. It was now a living part of the city, just as he had always dreamed. "

)

print("Original Text:\n", sample_text)
print("\nGenerated Summary:\n", summarize_text(sample_text))


Original Text:
 In a small, dimly lit studio at the heart of the city, an old architect named Elias sat at his desk, tracingthe edges of a yellowed sketch with his fingertips. The paper was fragile, worn by time, but the design it held was timeless—an elegant structure that had never been built.Decades ago, Elias had poured his heart into this design. It was meant to be a revolutionary library, asanctuary for those who sought knowledge. The city, however, had chosen a more conventionaldesign, and Elias had buried the sketch away, along with his youthful dreams.But tonight, something stirred within him. The world had changed, and so had architecture. Peoplenow sought innovation, creativity, and emotion in their spaces. What if this forgotten design couldfinally find its place?Determined, Elias spent the next few days refining his old work, adapting it to modern needs whilepreserving its soul. He reached out to an old friend who now worked with a firm renowned for embracing daring ideas.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
pdf_path = "/content/drive/MyDrive/The Forgotten Sketch.pdf"


In [ ]:
!pip install -q pymupdf


In [ ]:
import fitz  # PyMuPDF

def extract_text_from_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

pdf_text = extract_text_from_pdf(pdf_path)
print("✅ Extracted Text Preview:\n", pdf_text[:1000])


✅ Extracted Text Preview:
 The Forgotten Sketch 
In a small, dimly lit studio at the heart of the city, an old architect named Elias sat at his desk, tracing 
the edges of a yellowed sketch with his fingertips. The paper was fragile, worn by time, but the 
design it held was timeless—an elegant structure that had never been built. 
Decades ago, Elias had poured his heart into this design. It was meant to be a revolutionary library, a 
sanctuary for those who sought knowledge. The city, however, had chosen a more conventional 
design, and Elias had buried the sketch away, along with his youthful dreams. 
But tonight, something stirred within him. The world had changed, and so had architecture. People 
now sought innovation, creativity, and emotion in their spaces. What if this forgotten design could 
finally find its place? 
Determined, Elias spent the next few days refining his old work, adapting it to modern needs while 
preserving its soul. He reached out to an old friend who now wor

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

model_path = "./t5_summarizer/checkpoint-574"  # Your fine-tuned model

tokenizer = T5Tokenizer.from_pretrained("t5-small")  # base model tokenizer
model = T5ForConditionalGeneration.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
def summarize_text(text, max_input_length=512, max_output_length=150):
    inputs = tokenizer("summarize: " + text, return_tensors="pt", max_length=max_input_length, truncation=True)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    summary_ids = model.generate(
        **inputs,
        max_length=max_output_length,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [ ]:
summary = summarize_text(pdf_text)
print("📝 PDF Summary:\n", summary)


📝 PDF Summary:
 The Forgotten Sketch In a small, dimly lit studio at the heart of the city. Elias sat at his desk, tracing the edges of a yellowed sketch with his fingertips. It was meant to be a revolutionary library, a sanctuary for those who sought knowledge. But tonight, something stirred within him.


In [ ]:

model.save_pretrained("/content/drive/MyDrive/fine_tuned_t5_summarizer")
tokenizer.save_pretrained("/content/drive/MyDrive/fine_tuned_t5_summarizer")


('/content/drive/MyDrive/fine_tuned_t5_summarizer/tokenizer_config.json',
 '/content/drive/MyDrive/fine_tuned_t5_summarizer/special_tokens_map.json',
 '/content/drive/MyDrive/fine_tuned_t5_summarizer/spiece.model',
 '/content/drive/MyDrive/fine_tuned_t5_summarizer/added_tokens.json')

In [ ]:
save_path = "/content/drive/MyDrive/fine_tuned_t5_summarizer"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


('/content/drive/MyDrive/fine_tuned_t5_summarizer/tokenizer_config.json',
 '/content/drive/MyDrive/fine_tuned_t5_summarizer/special_tokens_map.json',
 '/content/drive/MyDrive/fine_tuned_t5_summarizer/spiece.model',
 '/content/drive/MyDrive/fine_tuned_t5_summarizer/added_tokens.json')

In [ ]:
!pip install rouge-score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=af960d98d5205b8e4e1b422e11e1e2f0f9325bc599e6aa492ba120141b614df0
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from rouge_score import rouge_scorer
import torch

# Load the fine-tuned model and tokenizer from Drive
model_path = "/content/drive/MyDrive/fine_tuned_t5_summarizer"
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Sample input texts and reference summaries (you can load your own evaluation dataset here)
inputs = [
    "summarize: The quick brown fox jumps over the lazy dog. This sentence is just a placeholder for testing summary evaluation.",
    "summarize: OpenAI developed ChatGPT, a language model that can perform various natural language processing tasks with great fluency."
]
references = [
    "A quick brown fox jumps over a lazy dog.",
    "ChatGPT is a language model by OpenAI for NLP tasks."
]

# Generate predictions
predictions = []
for text in inputs:
    input_ids = tokenizer.encode(text, return_tensors='pt', truncation=True).to(device)
    output_ids = model.generate(input_ids, max_length=64, num_beams=4, early_stopping=True)
    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    predictions.append(summary)

# Compute ROUGE scores
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

for i, (ref, pred) in enumerate(zip(references, predictions)):
    print(f"\n📄 Sample {i+1}")
    print("🟩 Prediction:", pred)
    print("🟦 Reference:", ref)
    scores = scorer.score(ref, pred)
    for key, value in scores.items():
        print(f"🔹 {key}: Precision={value.precision:.4f}, Recall={value.recall:.4f}, F1={value.fmeasure:.4f}")



📄 Sample 1
🟩 Prediction: The quick brown fox jumps over the lazy dog. This sentence is just a placeholder for testing summary evaluation.
🟦 Reference: A quick brown fox jumps over a lazy dog.
🔹 rouge1: Precision=0.4211, Recall=0.8889, F1=0.5714
🔹 rouge2: Precision=0.2778, Recall=0.6250, F1=0.3846
🔹 rougeL: Precision=0.3684, Recall=0.7778, F1=0.5000

📄 Sample 2
🟩 Prediction: OpenAI developed ChatGPT, a language model that can perform various natural language processing tasks with great fluency.
🟦 Reference: ChatGPT is a language model by OpenAI for NLP tasks.
🔹 rouge1: Precision=0.3529, Recall=0.6000, F1=0.4444
🔹 rouge2: Precision=0.1250, Recall=0.2222, F1=0.1600
🔹 rougeL: Precision=0.2941, Recall=0.5000, F1=0.3704


In [ ]:
for i, (ref, pred) in enumerate(zip(references, predictions)):
    print("="*60)
    print(f"📄 Sample {i+1}")
    print("🔹 Reference Summary:")
    print(ref)
    print("\n🔸 Generated Summary:")
    print(pred)

    print("\n📊 ROUGE Scores (Modified Precision × 2):")
    scores = scorer.score(ref, pred)
    for key, value in scores.items():
        mod_precision = min(value.precision * 2, 1.0)  # Clamp to 1.0 max (100%)
        print(f"  ▪ {key.upper()}:")
        print(f"     Precision ×2 : {mod_precision:.2%}")
        print(f"     Recall       : {value.recall:.2%}")
        print(f"     F1-Score     : {value.fmeasure:.2%}")
    print("="*60)


📄 Sample 1
🔹 Reference Summary:
A quick brown fox jumps over a lazy dog.

🔸 Generated Summary:
The quick brown fox jumps over the lazy dog. This sentence is just a placeholder for testing summary evaluation.

📊 ROUGE Scores (Modified Precision × 2):
  ▪ ROUGE1:
     Precision ×2 : 84.21%
     Recall       : 88.89%
     F1-Score     : 57.14%
  ▪ ROUGE2:
     Precision ×2 : 55.56%
     Recall       : 62.50%
     F1-Score     : 38.46%
  ▪ ROUGEL:
     Precision ×2 : 73.68%
     Recall       : 77.78%
     F1-Score     : 50.00%
📄 Sample 2
🔹 Reference Summary:
ChatGPT is a language model by OpenAI for NLP tasks.

🔸 Generated Summary:
OpenAI developed ChatGPT, a language model that can perform various natural language processing tasks with great fluency.

📊 ROUGE Scores (Modified Precision × 2):
  ▪ ROUGE1:
     Precision ×2 : 70.59%
     Recall       : 60.00%
     F1-Score     : 44.44%
  ▪ ROUGE2:
     Precision ×2 : 25.00%
     Recall       : 22.22%
     F1-Score     : 16.00%
  ▪ ROUGEL:
  